# Beijing Multi-Site Air Quality — Aotizhongxin station processing

Source: UCI ML Repository, "Beijing Multi-Site Air-Quality Data" (PRSA2017). 12 monitoring stations included in the archive; using a single station (`Aotizhongxin`, a commonly-used representative station in the literature) to keep one time series, consistent with every other candidate in this study.

`station` is constant (single value) and `No` is a row index — both dropped, not counted toward the feature minimum. `wd` (16-point compass wind direction) is categorical -> cyclical `sin`/`cos` encoding, since the network needs numeric input and compass direction wraps around (N is adjacent to both NNE and NNW, a plain integer/degree encoding would not reflect that).

In [ ]:
import sys
sys.path.append(".")
import pandas as pd
import numpy as np
from common import report_candidate

RAW_PATH = "../data/raw/beijing_air_quality/PRSA_Data_20130301-20170228/PRSA_Data_Aotizhongxin_20130301-20170228.csv"
PROCESSED_PATH = "../data/processed/beijing_air_quality_aotizhongxin.csv"

df = pd.read_csv(RAW_PATH)
df["timestamp"] = pd.to_datetime(dict(year=df.year, month=df.month, day=df.day, hour=df.hour))
df = df.set_index("timestamp").drop(columns=["No", "year", "month", "day", "hour", "station"])
df.isna().sum()

In [ ]:
COMPASS = {
    "N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5, "E": 90, "ESE": 112.5, "SE": 135, "SSE": 157.5,
    "S": 180, "SSW": 202.5, "SW": 225, "WSW": 247.5, "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5,
}
wd_deg = df["wd"].map(COMPASS)
df["wd_sin"] = np.sin(np.deg2rad(wd_deg))
df["wd_cos"] = np.cos(np.deg2rad(wd_deg))
df = df.drop(columns=["wd"])

TARGET = "PM2.5"
feature_cols = [c for c in df.columns if c != TARGET]
feature_cols

## Gaps

Pollutant sensors (`CO`, `O3` worst at ~5%) have more missing readings than the weather columns. Small gaps (<= `MAX_GAP_HOURS`) are interpolated; longer ones are dropped rather than filled.

**Important for the windowing step in Phase 1:** dropping unresolved gaps leaves the hourly index no longer perfectly contiguous (~0.3% of consecutive steps are wider than 1h after cleaning — check the `report_candidate` output below). A sliding window must not be built across one of these gaps. Either (a) reindex to the full hourly grid first and treat the gap as a hard split point per contiguous block, or (b) check consecutive timestamp deltas while generating windows and skip any window whose span isn't exactly `window_length` hours. Do this before Phase 1, not after seeing bad results.

In [ ]:
MAX_GAP_HOURS = 3
before = len(df)
df = df.interpolate(method="time", limit=MAX_GAP_HOURS).dropna(how="any")
print(f"dropped {before - len(df)} rows with unresolved gaps")

In [ ]:
report_candidate(df, TARGET, feature_cols, freq="1h", name="Beijing Air Quality - Aotizhongxin (processed)")

In [ ]:
df.to_csv(PROCESSED_PATH)
print(f"saved: {PROCESSED_PATH}  shape={df.shape}")